# Lab 19-04 · Logical Effort: Sizing Paths and Buffer Chains

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize
import effort as ef
TAU = 0.6931471805599453 * 12.57e3 * 2e-15   # 19-02's inverter, 17.43 ps

## Part 1 · Buffer chains at transistor level

*Predict* the delays of a unit inverter and of a 1-4-16 chain driving 128 fF (H = 64) from logical effort, then compare with `ngspice -b chain.cir`.

In [2]:
t1 = ef.simulate_chain([1], 128e-15)
t3 = ef.simulate_chain([1, 4, 16], 128e-15)
print(f"single inverter: {t1 * 1e12:.1f} ps = {t1 / TAU:.1f} tau (logical effort: 65 tau)")
print(f"three stages:    {t3 * 1e12:.1f} ps = {t3 / TAU:.1f} tau (logical effort: 15 tau)")
print(f"speed-up {t1 / t3:.2f} (logical effort: {65 / 15:.2f})")

single inverter: 1095.9 ps = 62.9 tau (logical effort: 65 tau)
three stages:    404.3 ps = 23.2 tau (logical effort: 15 tau)
speed-up 2.71 (logical effort: 4.33)


## Part 2 · How many stages?

*Predict* the best of one to four stages for H = 64 at transistor level.

In [3]:
for n in (1, 2, 3, 4):
    sizes = [64 ** (i / n) for i in range(n)]
    t = ef.simulate_chain(sizes, 128e-15)
    print(f"{n} stages, sizes {[round(s, 2) for s in sizes]}: {t * 1e12:6.1f} ps ({t / TAU:5.1f} tau; model {ef.chain_delay(64, n):5.2f})")

1 stages, sizes [1.0]: 1095.9 ps ( 62.9 tau; model 65.00)
2 stages, sizes [1.0, 8.0]:  437.0 ps ( 25.1 tau; model 18.00)
3 stages, sizes [1.0, 4.0, 16.0]:  404.3 ps ( 23.2 tau; model 15.00)
4 stages, sizes [1.0, 2.83, 8.0, 22.63]:  433.1 ps ( 24.9 tau; model 15.31)


## Part 3 · Equal efforts, found numerically

*Predict* the sizes that minimize Example 19-04.3's delay (NAND2 with input 10, branching 2, NOR2, inverter, load 200).

In [4]:
g = [4 / 3, 5 / 3, 1]; p = [2, 2, 1]; b = [2, 1, 1]
def total(logs):
    s = [10 / g[0]] + list(np.exp(logs))   # drive of each gate; input capacitance g * s
    d = 0.0
    for i in range(3):
        nxt = g[i + 1] * s[i + 1] if i < 2 else 200
        d += p[i] + b[i] * nxt / s[i]
    return d
res = minimize(total, np.log([5, 30]), method="Nelder-Mead", options={"xatol": 1e-10, "fatol": 1e-12})
s = [10 / g[0]] + list(np.exp(res.x))
print("numerical: delay", round(res.fun, 4), "tau; input capacitances", [round(g[i] * s[i], 2) for i in range(3)])
print("logical effort:", round(ef.min_path_delay(g, p, b, 20), 4), "tau;", [round(c, 2) for c in ef.stage_input_caps(g, b, 10, 200)])

numerical: delay 18.3887 tau; input capacitances [10.0, np.float64(16.74), np.float64(44.81)]
logical effort: 18.3887 tau; [10.0, 16.74, 44.81]


## Part 4 · The best stage effort

*Predict* how rho changes as the inverter's parasitic delay grows from 0 to 3.

In [5]:
pinvs = np.linspace(0, 3, 31)
plt.plot(pinvs, [ef.best_stage_effort(x) for x in pinvs]); plt.xlabel("p_inv"); plt.ylabel("best stage effort rho"); plt.show()
for x in (0, 1, 2, 3):
    print(f"p_inv = {x}: rho = {ef.best_stage_effort(x):.3f}")
f = np.linspace(2, 8, 61)
per = (f + 1) / np.log(f)
print("within 9% of the minimum for f from", round(f[per <= 1.09 * per.min()].min(), 2), "to", round(f[per <= 1.09 * per.min()].max(), 2))

p_inv = 0: rho = 2.718
p_inv = 1: rho = 3.591
p_inv = 2: rho = 4.319
p_inv = 3: rho = 4.971
within 9% of the minimum for f from 2.4 to 6.0


## Part 5 · Topologies

*Predict* which 8-input AND topology wins as H goes from 1 to 1000.

In [6]:
h = np.logspace(0, 3, 61)
cases = {"NAND8+INV": ([10 / 3, 1], [8, 1]), "NAND4+NOR2": ([2, 5 / 3], [4, 2]), "tree": ([4 / 3, 5 / 3, 4 / 3, 1], [2, 2, 2, 1])}
for name, (gs, ps) in cases.items():
    plt.semilogx(h, [ef.min_path_delay(gs, ps, [1] * len(gs), x) for x in h], label=name)
plt.xlabel("H"); plt.ylabel("delay (tau)"); plt.legend(); plt.show()
for x in (1, 10, 100):
    print(x, {k: round(ef.min_path_delay(gs, ps, [1] * len(gs), x), 2) for k, (gs, ps) in cases.items()})

1 {'NAND8+INV': 12.65, 'NAND4+NOR2': 9.65, 'tree': 12.25}
10 {'NAND8+INV': 20.55, 'NAND4+NOR2': 17.55, 'tree': 16.33}
100 {'NAND8+INV': 45.51, 'NAND4+NOR2': 42.51, 'tree': 23.6}


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).